# Pressure-Weighted DPO — Sycophancy Resistance Training

Trains **Qwen2-VL-7B-Instruct** with **pressure-weighted DPO** on the sycophancy dataset.  
Higher-pressure examples (p3, p4) receive proportionally more gradient signal.

Loss weighting: `p1→0.1 | p2→0.2 | p3→0.3 | p4→0.4 | B/C/D→1.0`

Checkpoints are saved to Google Drive. W&B gives a live loss curve link.

> **Runtime**: Runtime → Change runtime type → **A100 GPU**  
> **Parallel run**: This notebook runs alongside `colab_standard_dpo.ipynb` in a separate Colab tab.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_SAVE_DIR = '/content/drive/MyDrive/sycophancy_dpo/weighted'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_SAVE_DIR}')

## 2. Install Dependencies

In [ ]:
%%capture install_output
!pip install \
    transformers==4.45.2 \
    trl==0.11.4 \
    peft==0.12.0 \
    accelerate==0.34.2 \
    wandb \
    datasets \
    matplotlib -q
print('Installation complete.')

## 3. Clone Repo & Verify Dataset

In [ ]:
import os

REPO_URL = 'https://github.com/rupalirajesh/sycophancy_VidLMs.git'
REPO_DIR = '/content/sycophancy_VidLMs'

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print('Repo ready at:', REPO_DIR)

In [ ]:
import json, os

TRAIN_PATH = f'{REPO_DIR}/output/train.jsonl'
VAL_PATH   = f'{REPO_DIR}/output/val.jsonl'

for path in [TRAIN_PATH, VAL_PATH]:
    size_mb = os.path.getsize(path) / 1e6
    with open(path) as f:
        n = sum(1 for _ in f)
    print(f'{os.path.basename(path)}: {n:,} examples  ({size_mb:.1f} MB)')

# Show weight distribution (this is what makes weighted DPO different)
from collections import Counter
with open(TRAIN_PATH) as f:
    weights = Counter(json.loads(l)['loss_weight'] for l in f)
print('\nloss_weight distribution:')
for w, cnt in sorted(weights.items()):
    print(f'  {w:.1f} → {cnt:,} examples')

## 4. W&B Login
Get your API key from https://wandb.ai/authorize

In [ ]:
import wandb
wandb.login()  # paste API key when prompted

## 5. Configure Output Paths

In [ ]:
CHECKPOINT_DIR = f'{REPO_DIR}/checkpoints/weighted-dpo-qwen2vl'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print('Training checkpoints →', CHECKPOINT_DIR)
print('Google Drive backup  →', DRIVE_SAVE_DIR)

!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

## 6. Train — Pressure-Weighted DPO

Expected time: ~4–6 hours on A100.  
Click the W&B link printed below to watch live loss curves.  

The `--weighted` flag activates `WeightedDPOTrainer`, which scales each sample's  
loss by its `loss_weight` before the `.mean()` reduction.

In [ ]:
!python train_dpo.py \
    --train {TRAIN_PATH} \
    --val   {VAL_PATH} \
    --output-dir {REPO_DIR}/checkpoints \
    --weighted \
    --epochs 2 \
    --batch-size 2 \
    --grad-accum 8 \
    --lr 5e-5 \
    --beta 0.1 \
    --max-length 1024

## 7. Save to Google Drive

In [ ]:
import shutil, os

print('Copying checkpoints to Drive...')
if os.path.isdir(DRIVE_SAVE_DIR):
    shutil.rmtree(DRIVE_SAVE_DIR)
shutil.copytree(CHECKPOINT_DIR, DRIVE_SAVE_DIR)

total = 0
for root, dirs, files in os.walk(DRIVE_SAVE_DIR):
    for f in files:
        total += os.path.getsize(os.path.join(root, f))
print(f'Saved {total/1e9:.2f} GB to {DRIVE_SAVE_DIR}')

## 8. Plot Loss Curve

In [ ]:
import json, matplotlib.pyplot as plt, matplotlib.ticker as mticker

log_path = f'{CHECKPOINT_DIR}/loss_log.json'

with open(log_path) as f:
    entries = json.load(f)

train_steps = [e['step'] for e in entries if 'train_loss' in e]
train_loss  = [e['train_loss'] for e in entries if 'train_loss' in e]
eval_steps  = [e['step'] for e in entries if 'eval_loss' in e]
eval_loss   = [e['eval_loss']  for e in entries if 'eval_loss' in e]
rm_steps    = [e['step'] for e in entries if 'reward_margin' in e]
rm_vals     = [e['reward_margin'] for e in entries if 'reward_margin' in e]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Pressure-Weighted DPO — Training Progress', fontsize=13, fontweight='bold')

axes[0].plot(train_steps, train_loss, color='#F44336', linewidth=1.5, label='train')
axes[0].set_title('Train Loss (weighted)'); axes[0].set_xlabel('Step'); axes[0].grid(alpha=0.3)

axes[1].plot(eval_steps, eval_loss, color='#FF9800', linewidth=1.5, linestyle='--', label='eval')
axes[1].set_title('Eval Loss'); axes[1].set_xlabel('Step'); axes[1].grid(alpha=0.3)

axes[2].plot(rm_steps, rm_vals, color='#9C27B0', linewidth=1.5)
axes[2].set_title('Reward Margin (chosen − rejected)'); axes[2].set_xlabel('Step'); axes[2].grid(alpha=0.3)

for ax in axes:
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plot_path = f'{DRIVE_SAVE_DIR}/loss_curve.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f'Plot saved to {plot_path}')
plt.show()

## 9. (Optional) Compare Both Runs

Run this cell **after both notebooks have finished** to overlay standard vs weighted loss curves.

In [ ]:
# Load both logs from Drive
import json, matplotlib.pyplot as plt, matplotlib.ticker as mticker

def load_log(path):
    with open(path) as f:
        return json.load(f)

std_log  = load_log('/content/drive/MyDrive/sycophancy_dpo/standard/loss_log.json')
wtd_log  = load_log('/content/drive/MyDrive/sycophancy_dpo/weighted/loss_log.json')

def extract(log, key):
    return zip(*[(e['step'], e[key]) for e in log if key in e]) or ([], [])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Standard vs Weighted DPO Comparison', fontsize=13, fontweight='bold')

for ax, key, title in zip(axes, ['train_loss', 'eval_loss', 'reward_margin'],
                           ['Train Loss', 'Eval Loss', 'Reward Margin']):
    for log, label, color in [(std_log, 'Standard DPO', '#2196F3'), (wtd_log, 'Weighted DPO', '#F44336')]:
        pairs = [(e['step'], e[key]) for e in log if key in e]
        if pairs:
            xs, ys = zip(*pairs)
            ax.plot(xs, ys, color=color, label=label, linewidth=1.5)
    ax.set_title(title); ax.set_xlabel('Step'); ax.legend(fontsize=9); ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
compare_path = '/content/drive/MyDrive/sycophancy_dpo/comparison.png'
plt.savefig(compare_path, dpi=150, bbox_inches='tight')
print(f'Comparison plot saved to {compare_path}')
plt.show()